# **Distinct HCP NPIs from CRM Calls**
- Extracts unique HCP NPIs from CRM call activity within a defined time period
- Uses VCRM call data as the base interaction dataset
- Joins with customer master to map account IDs to HCP NPIs
- Filters records for Start and End Date and removes null NPIs
- Ensures deduplication using DISTINCT
- Output used for HCP engagement tracking and downstream analytics

In [0]:
%sql

WITH call_hcps AS (
    SELECT DISTINCT
        TRY_CAST(c.npi__v AS STRING) AS npi,
        a.account__v,
        a.attendee_type__v,
        a.detailed_products__v
    FROM com_edp_prd.com_raw.vcrm_call2__v a
    LEFT JOIN com_intgr.customer c
        ON a.account__v = c.id
    WHERE a.call_date__v BETWEEN '2026-03-01' AND '2026-04-30'
      AND a.call2_status__v = 'submitted__v'  
      AND c.npi__v IS NOT NULL
),

reference_hcps AS (
    SELECT DISTINCT
        TRY_CAST(hcp_npi AS STRING) AS npi
    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703
),
base AS (
SELECT DISTINCT
    ch.npi,
    ch.attendee_type__v,
    ch.account__v,
    ch.detailed_products__v,
    COALESCE(concat(p.FIRST_NAME," ", p.LAST_NAME), p.ORGANIZATION_NAME) as name,
    CASE 
        WHEN rh.npi IS NOT NULL THEN 1 
        ELSE 0 
    END AS is_in_reference_file
FROM call_hcps ch
LEFT JOIN reference_hcps rh
    ON ch.npi = rh.npi
LEFT JOIN com_edp_prd.com_raw.kom_providers p
    ON ch.npi = P.NPI
)

SELECT * from base 
where is_in_reference_file != 1
and attendee_type__v = 'person_account__v'
-- and detailed_products__v = 'AVLAYAH'

;

#  HCP Engagement Flagging on Reference File
- Creates a temp view to flag whether HCPs from the reference file were engaged via CRM calls
- Identifies engaged HCPs by matching call activity NPIs with reference file NPIs
- Uses VCRM call data filtered for March 2026 as the engagement source
- INNER JOIN ensures only NPIs present in both call data and reference file are considered “engaged”
- LEFT JOIN back to full reference file to retain all HCPs
- Adds binary flag is_hcp_engaged → 1 = HCP had at least one call interaction & 0 = No interaction in given period
- Output used for coverage analysis, targeting, and engagement tracking

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_reference_0316_hcp_engaged AS

WITH hcp_engaged AS (
  SELECT DISTINCT
    TRY_CAST(c.npi__v AS STRING) AS hcp_npi
  FROM com_edp_prd.com_raw.vcrm_call2__v a
  LEFT JOIN com_intgr.customer c
    ON a.account__v = c.id
  INNER JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703 r
    ON TRY_CAST(c.npi__v AS STRING) = TRY_CAST(r.hcp_npi AS STRING)
  WHERE a.call_date__v BETWEEN '2026-03-01' AND '2026-03-31'   --Will be variable to as per our analysis needs
)

SELECT
  r.*,
  CASE WHEN he.hcp_npi IS NOT NULL THEN 1 ELSE 0 END AS is_hcp_engaged
FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703 r
LEFT JOIN hcp_engaged he
  ON TRY_CAST(r.hcp_npi AS STRING) = he.hcp_npi
;

In [0]:
%sql
select * from temp_reference_0316_hcp_engaged

In [0]:
%sql
SELECT COUNT(DISTINCT hcp_npi) AS engaged_hcp_count
FROM temp_reference_0316_hcp_engaged
WHERE is_hcp_engaged = 1;

# Extract Engaged HCP NPIs (Filtered by Reference File)
- Identifies HCPs with CRM call activity within March 2026
- Maps call records to HCP NPIs using customer master
- Filters only those NPIs present in the reference file (target HCP universe)
- Ensures valid NPIs using type casting and null exclusion
- Deduplicates to get unique engaged HCP list

In [0]:
%sql
WITH hcp_engaged AS (
  SELECT DISTINCT
    TRY_CAST(c.npi__v AS STRING) AS hcp_npi
  FROM com_edp_prd.com_raw.vcrm_call2__v a
  LEFT JOIN com_intgr.customer c
    ON a.account__v = c.id
  INNER JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0316 r31
    ON TRY_CAST(c.npi__v AS STRING) = TRY_CAST(r31.hcp_npi AS STRING)
  WHERE a.call_date__v BETWEEN '2026-03-01' AND '2026-03-31'
)

SELECT DISTINCT hcp_npi
FROM hcp_engaged
WHERE hcp_npi IS NOT NULL
ORDER BY hcp_npi;

In [0]:
%sql
SELECT DISTINCT
    TRY_CAST(c.npi__v AS STRING) AS hcp_npi,
    c.hcp_facing_display_name__v
FROM com_edp_prd.com_raw.vcrm_call2__v a
LEFT JOIN com_intgr.customer c
    ON a.account__v = c.id
WHERE a.call_date__v BETWEEN '2026-03-01' AND '2026-03-31'
  AND TRY_CAST(c.npi__v AS STRING) IS NOT NULL
  AND c.hcp_facing_display_name__v IS NOT NULL
ORDER BY hcp_npi;

In [0]:
select distinct dnli_hs_patients__c,
dnli_hunter_syndrome_pts_primary_parent__c,
dnli_of_hunter_syndrome_patients__c,
dnli_tivi_patients__c from com_edp_prd.com_raw.vcrm_account__v



In [0]:
select * FROM com_edp_prd.com_raw.kom_medical_events
WHERE (PROCEDURE_CODE IN ('J3490','J3590','J9999') AND SERVICE_DATE >= '2026-03-01')
OR NDC11 IN ('8497600101')


In [0]:
select * FROM com_edp_prd.com_raw.kom_pharmacy_events limit 5;
-- WHERE NDC11 IN ('8497600101')
